# Loan Default Prediction Notebook
This notebook loads the dataset, trains a model, saves a pickle file, creates a test CSV, and launches a simple UI for custom predictions.

In [ ]:
import pandas as pd, numpy as np, os, joblib, warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import BaseEstimator, ClassifierMixin

try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = False

DATA_PATH = 'idbi_loan_default_hackathon_dataset_15000.csv'
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
target = 'loan_default'
drop_cols = ['loan_id','customer_id','disbursal_date']
X = df.drop(columns=[target] + [c for c in drop_cols if c in df.columns])
y = df[target].astype(int)
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()
preprocess = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
test_df = X_test.copy()
test_df[target] = y_test.values
test_df.to_csv('test.csv', index=False)
print('Train:', X_train.shape, 'Test:', X_test.shape, 'Cats:', len(cat_cols), 'Nums:', len(num_cols))

In [ ]:
class WrappedModel(BaseEstimator, ClassifierMixin):
    def __init__(self, model, preprocess):
        self.model = model
        self.preprocess = preprocess
    def fit(self, X, y):
        Xp = self.preprocess.fit_transform(X)
        self.model.fit(Xp, y)
        return self
    def predict(self, X):
        return self.model.predict(self.preprocess.transform(X))
    def predict_proba(self, X):
        return self.model.predict_proba(self.preprocess.transform(X))

# Class Balancing: The dataset is highly imbalanced (~93.8% Default cases).
# We perform Random Undersampling on the training set to prevent model bias.
df_train = X_train.copy()
df_train[target] = y_train.values
df_class_0 = df_train[df_train[target] == 0]
df_class_1 = df_train[df_train[target] == 1]

# Downsample class 1 to match class 0 size
df_class_1_down = df_class_1.sample(n=len(df_class_0), random_state=42)
df_balanced = pd.concat([df_class_0, df_class_1_down]).sample(frac=1, random_state=42)

X_train_balanced = df_balanced.drop(columns=[target])
y_train_balanced = df_balanced[target].astype(int)
print('Balanced training set size:', X_train_balanced.shape)

if HAS_XGB:
    model = xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.08, subsample=0.9, colsample_bytree=0.85, eval_metric='logloss', random_state=42)
else:
    model = RandomForestClassifier(n_estimators=300, max_depth=None, random_state=42, n_jobs=-1)

pipe = WrappedModel(model, preprocess)
pipe.fit(X_train_balanced, y_train_balanced)

pred = pipe.predict(X_test)
proba = pipe.predict_proba(X_test)[:,1] if hasattr(pipe.predict_proba(X_test), '__len__') else None
acc = accuracy_score(y_test, pred)
auc = roc_auc_score(y_test, proba) if proba is not None else None
print('Accuracy on original Test Set:', round(acc,4))
print('AUC-ROC on original Test Set:', round(auc,4) if auc is not None else 'NA')
print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))
joblib.dump(pipe, 'loan_default_model.pkl')
print('Saved balanced model to: loan_default_model.pkl')

In [ ]:
sample = X_test.iloc[[0]].copy()
sample.to_csv('sample_custom_input.csv', index=False)
sample.to_dict(orient='records')[0]

## Simple UI
Run the cell below in Jupyter/Colab to create a small interface with your saved pickle model. If Streamlit is preferred, I can generate a separate app.py too.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    model = joblib.load('loan_default_model.pkl')

    widgets_dict = {}
    for col in X.columns:
        if col in cat_cols:
            options = sorted([str(v) for v in df[col].dropna().astype(str).unique().tolist()])[:100]
            w = widgets.Dropdown(options=options, description=col[:18], layout=widgets.Layout(width='420px'))
        else:
            mn = float(pd.to_numeric(df[col], errors='coerce').min())
            mx = float(pd.to_numeric(df[col], errors='coerce').max())
            val = float(pd.to_numeric(df[col], errors='coerce').median())
            step = 1.0 if mx - mn < 100 else max(1.0, (mx-mn)/100)
            w = widgets.FloatText(value=val, description=col[:18], layout=widgets.Layout(width='420px'))
        widgets_dict[col] = w

    btn = widgets.Button(description='Predict', button_style='success')
    out = widgets.Output()

    def on_click(_):
        row = {}
        for c, w in widgets_dict.items():
            row[c] = w.value
        inp = pd.DataFrame([row])
        p = model.predict(inp)[0]
        pr = model.predict_proba(inp)[0,1]
        with out:
            clear_output()
            print('Prediction:', int(p))
            print('Default probability:', round(float(pr), 4))

    btn.on_click(on_click)
    display(widgets.VBox([widgets_dict[c] for c in X.columns[:18]] + [btn, out]))
except Exception as e:
    print('UI not available in this environment:', e)